In [10]:
import os, sys
from os import listdir
from os.path import isfile, join
import numpy as np
from scipy import stats
from scipy.stats import f_oneway
from numpy import array
import pandas as pd
import geopandas as gp
# from gisutils import project
from shapely.geometry import Point
import statistics
import math
from math import log10, floor
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
import glob
from pyproj import Transformer
from glob import glob
import xarray as xr
from copy import deepcopy as dc
import matplotlib.gridspec as gridspec
import matplotlib.colors as colors
import matplotlib.cm as cmx
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogFormatterMathtext
from matplotlib.animation import FuncAnimation
from rasterio.mask import mask
import rioxarray as rio

def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name,a=minval,b=maxval),
        cmap(np.linspace(minval, maxval,n)))
    return new_cmap

testingpguL={'col1':[-448.914,-440,-441.96,-448.914,-449.9,-450],'condition':[2,2,3,4,5]}
df=pd.DataFrame(data=testingpguL)

df.loc[:,'nearest10']=-9999
df['roundgz'] =-9999
df['roundgzint'] =-9999
df['numstring'] =-9999
df['ld'] =-9999
df['ldan'] =-9999

df['roundgz'] = df['col1'].round()
df['roundgzint'] = df['roundgz'].astype(int)
df['nearest10'] = df['col1'].round(-1)
df['numstring'] = df['roundgzint'].astype(str)
df['ld'] = df['numstring'].str[-1]
df['ldan'] = df['ld'].astype(int)

# Calculate rgzbotm based on conditions
    df['rgzbotm'] = np.where(df['col1'] > 0,
                            df['roundgzint'] - df['ldan'],
                            df['roundgzint'] + df['ldan'])
    
    condition1 = (df['GlobalZ_m'] >= -450) & (df['ld'].isin([8, 9])) & (df['GlobalZ_m'] > 0)

    # Assign values based on conditions
    df.loc[condition1, 'topelv1'] = df['nearest10int']
    df.loc[condition1, 'botelv1'] = df['topelv1'] - 5


print('roundgz:',df['roundgz'][0])
print('roundgzint:',df['roundgzint'][0])
print('nearest10:',df['nearest10'][0])
print('numstring:',df['numstring'][0])
print('ld:',df['ld'][0])
print('ldan:',df['ldan'][0])

In [11]:
# path to file
#datafile = os.path.join('../','MAP_RegionalAEM_2020_ResistivityFacies_DepthGrids.nc')
#datafile = os.path.join('SHMD_krig_res_depth_02m.nc')

# read in as xarray dataset
#ds = xr.load_dataset(datafile)


In [12]:
# Get list of pathline files in the specified directory
#pathline_dir = r'C:\Users\chaugh\Documents\micheal_g_files'
#pathline_dir = r'C:\github\map_gwage\MERASwd\Scripts\meras\testFILZ'
#pathline_dir = r'C:\github\map_gwage\MERASwd\Scripts\meras'
# pathline_dir = r'C:\github\map_gwage\MERASwd\Models\meras_2.2\layers_cal_wt_1.00\meras1\remaining'
# meras_files = glob.glob(os.path.join(pathline_dir, '*.mppth'))
# #pathline_dir = '../../../pathlineFiles8294'
# #meras_files = [f for f in os.listdir(pathline_dir) if os.path.isfile(os.path.join(pathline_dir, f))]
# firstpathline_dir = r'C:\github\map_gwage\MERASwd\Models\meras_2.2\layers_cal_wt_1.00\meras1\remaining'
secondpathline_dir = r'C:\github\map_gwage\MERASwd\Models\meras_2.2\layers_cal_wt_1.00\meras1'
# thirdpathline_dir=r'C:\github\map_gwage\MERASwd\Models\meras_2.2\layers_cal_wt_1.00\meras1\Shellmound'
#fourthpathline_dir=r'C:\github\map_gwage\MERASwd\Scripts\meras'
#fourthpathline_dir=r'C:\github\map_gwage\meraswd\Models\meras_2.2\layers_cal_wt_1.00\meras1\remaining'
# meras_files = glob.glob(os.path.join(firstpathline_dir, '*.mppth'))
# meras_files2 = glob.glob(os.path.join(secondpathline_dir, '*.mppth'))
# meras_files3 = glob.glob(os.path.join(thirdpathline_dir, '*.mppth'))
# meras_files4 = glob.glob(os.path.join(fourthpathline_dir, '*.mppth'))
# #final6=glob.glob(os.path.join(fourthpathline_dir,['meras_2.2_volume_zones[335308090362102].mppth','meras_2.2_volume_zones[340740091211501].mppth', 'meras_2.2_volume_zones[340905091495201].mppth', 'meras_2.2_volume_zones[342207090373201].mppth', 'meras_2.2_volume_zones[342553091225101].mppth', 'meras_2.2_volume_zones[343014091325401].mppth', 'meras_2.2_volume_zones[343015091325401].mppth']))
# final6=glob.glob(os.path.join(fourthpathline_dir,'last6','*.mppth'))
# Initialize variables
#sitenos = [f[-22:-7] for f in meras_files]  # Extract site number from file names. I think 22 is not included but 7 is, i.e., (22:7]
#num_files = len(meras_files)
#sitenos2 = [f[-22:-7] for f in meras_files2]  # Extract site number from file names
#num_files = len(meras_files2)
#sitenos3 = [f[-22:-7] for f in meras_files3]  # Extract site number from file names
#num_files = len(meras_files3)
# sitenos4 = [f[-22:-7] for f in meras_files4]  # Extract site number from file names
# num_files = len(meras_files4)
# sitenosl6 = [f[-22:-7] for f in final6]  # Extract site number from file names
# num_files = len(final6)
# print('Number of files:', num_files)

# Create DataFrame from file names and site numbers
#merasdf = pd.DataFrame({'indexpl': range(num_files), 'namepl': meras_files, 'siteno': sitenos})
#merasdf2 = pd.DataFrame({'indexpl': range(num_files), 'namepl': meras_files2, 'siteno': sitenos2})
#merasdf3 = pd.DataFrame({'indexpl': range(num_files), 'namepl': meras_files3, 'siteno': sitenos3})
#merasdf4 = pd.DataFrame({'indexpl': range(num_files), 'namepl': meras_files4, 'siteno': sitenos4})
#merasdfl6 = pd.DataFrame({'indexpl': range(num_files), 'namepl': final6, 'siteno': sitenosl6})
#merasdfl6=pd.read_csv('merasdfl6pguL.csv')
#merasdfl6=pd.read_csv('onemoreshot953.csv')
#merasdfl6=pd.read_csv('first2_1moshot.csv')
#merasdfl6=pd.read_csv('last_1mo.csv')
#merasdfl6=pd.read_csv('trynnnnn.csv')
#merasdfl6=pd.read_csv('try3real.csv')
merasdfl6=pd.read_csv('well35074.csv')
sitenolist=[]
numsparts=[]
MERs=[]
VERs=[]
AVRs=[]
VVRs=[]
cvers=[]
acvrs=[]
VcVRs=[]
cvcvrs=[]
logNormalityWell=[]
# Loop through each pathline file
for idx, row in merasdfl6.iterrows():
    filename = row['namepl']
    print('filename:',filename)
    #filename = 'meras_2.2_volume_zones[323757090515301].mppth'
    #filename = 'meras_2.2_volume_zones[320233091395501].mppth'
    #filename = 'meras_2.2_volume_zones[324044090323402].mppth'
    #filename = 'meras_2.2_volume_zones[333315090105302].mppth'
    siteno = row['siteno']
    print('siteno:',siteno)
    #siteno = 324044090323402
    #siteno = 333315090105302
    sitenolist.append(siteno)
    print('Processing file:', filename)

    # Define column names and read pathline file
    column_names = ['Particle ID', 'Particle group', 'Time Point Index', 'Cumulative Time Step', 'Tracking Time', 
                    'Global X', 'Global Y', 'Global Z', 'Layer', 'Row', 'Column', 'Grid', 
                    'Local X', 'Local Y', 'Local Z', 'Line Segment Index']
    #fourthpathline_dir=row['pathpl']
    #file_path = os.path.join(fourthpathline_dir, f"{filename}")
    file_path = os.path.join(secondpathline_dir, filename)
    #file_path = os.path.join('../meras', 'meras_2.2_volume_zones[324044090323402].mppth')
    #file_path = os.path.join('../meras', 'meras_2.2_volume_zones[333315090105302].mppth')
    #file_path = os.path.join('../../Models/meras_2.2/layers_cal_wt_1.00/meras1/remaining', 'meras_2.2_volume_zones[364520089383301].mppth')
    #df = pd.read_csv(file_path, skiprows=3, header=None, delim_whitespace=True, names=column_names)
    df = pd.read_csv(file_path, skiprows=3, header=None, sep='\s+', names=column_names)#df is a pathline file, representative of one flow path (particle), with each row representing a location along the flow path.
    numparts=max(df['Particle ID'])
    dfshape=df.shape
    numlocs=dfshape[0]
    print('numparts:',numparts)
    numsparts.append(numparts)
    # Convert coordinates from local to EPSG:5070
    xoff, yoff = 178389, 938511.6
    df['x_5070'] = xoff + df['Global X'] * 12 * 2.54 * 0.01
    df['y_5070'] = yoff + df['Global Y'] * 12 * 2.54 * 0.01

    # Reproject to EPSG:4326
    coords_5070 = np.array(list(zip(df['x_5070'], df['y_5070'])))
    #coords_4326 = project(coords_5070, 'epsg:5070', 'epsg:4326')
    #df['x_4326'], df['y_4326'] = coords_4326[:, 0], coords_4326[:, 1]
    #Transform the coordinates from 5070 to 4326
    transformer = Transformer.from_crs("EPSG:5070", "EPSG:4326", always_xy=True)
    df['x_4326'], df['y_4326'] = transformer.transform(df['x_5070'].values, df['y_5070'].values)

    print('Reprojection completed')
    print('max x_4326:',max(df['x_4326']))
    print('median x_4326:',statistics.median(df['x_4326']))
    print('mean x_4326:',np.mean(df['x_4326']))
    print('min x_4326:',min(df['x_4326']))
    print('max y_4326:',max(df['y_4326']))
    print('median y_4326:',statistics.median(df['y_4326']))
    print('mean y_4326:',np.mean(df['y_4326']))
    print('min y_4326:',min(df['y_4326']))
    #sys.exit()
    # Initialize new columns with default values
    #     default_values = {
    #         'number_of_locs':1,'dfmr':0,'dfmsr':0,'roundgz': -9999,'roundgzint':-9999,'nearest10': -9999,'nearest10int':-9999,'numstring': 'xx', 'ld': 'xx', 'ldan': -9999,
    #         'rgzbotm': -9999, 'topelv1': -9999, 'botelv1': -9999, 'topelv2': -9999, 'botelv2': -9999,
    #         'elevationSlice1': 'xx', 'elevationSlice2': 'xx', 'res_ohm_m1': -9999, 'res_ohm_m2': -9999,
    #         'log10res': -9999,'thickness':-9999, 'distance traveled': -9999, 'CumulativeDistanceTraveledIndex': -9999,
    #         'CumulativeDistanceTraveled': -9999, 'distance traveled within cell': -9999,
    #         'CumulativeDistanceTraveledwinCellIndex': -9999, 'CumulativeDistanceTraveledwinCell': -9999,
    #         'mpk': 0, 'mplogk': 0, 'mpsnk': 0, 'mpsnlogk': 0,'erdfm':0,'erdfms':0,'cellIndex': -9999, 'logNormality': 'xx'
    #     }
    #flow path segment-level values
    #default_values = {'siteno':'xx','pids':'xx','dfmr':-9999,'dfmsr':-9999,'roundgz': -9999,'roundgzint':-9999,'nearest10': -9999,'nearest10int':-9999,'numstring': 'xx', 'ld': 'xx', 'ldan': -9999,
    #         'rgzbotm': -9999, 'topelv1': -9999, 'botelv1': -9999, 'topelv2': -9999, 'botelv2': -9999,
    #         'elevationSlice1': 'xx', 'elevationSlice2': 'xx', 'res_ohm_m1': -9999, 'res_ohm_m2': -9999,
    #         'log10res': -9999,'thickness':-9999,'mpk': 0,'latintbot':-9999,'longintbot':-9999,'elvintbot':-9999,'latinttop':-9999,'longinttop':-9999,'elvinttop':-9999
    #     }
    default_values = {
        'siteno':'xx','pids':'xx','dfmr':-9999,'dfmsr':-9999,'roundgz': -9999,'roundgzint':-9999,'nearest10': -9999,'nearest10int':-9999,'numstring': 'xx', 'ld': 'xx', 'ldan': -9999,
        'rgzbotm': -9999, 'topelv1': -9999, 'botelv1': -9999, 'topelv2': -9999, 'botelv2': -9999,
        'elevationSlice1': 'xx', 'elevationSlice2': 'xx', 'res_ohm_m1': -9999, 'res_ohm_m2': -9999,
        'log10res': -9999,'thickness':-9999,'mpk': 0}

    for col, value in default_values.items():
        df[col] = value

    df['siteno'] = siteno
    df=df.replace(-9999,np.nan)
    plus1=df['dfmr'][0]+1
    print('plus1:',plus1)
    #sys.exit()
    # Start elevation extraction logic
    print('Start extraction.')

    # Convert Global Z to meters and compute rounding values
    df['GlobalZ_m'] = df['Global Z'] * 12 * 2.54 * 0.01
    df['roundgz'] = df['GlobalZ_m'].round()
    df['roundgzint'] = df['roundgz'].astype(int)
    df['nearest10'] = df['GlobalZ_m'].round(-1)
    df['nearest10int'] = df['nearest10'].astype(int)
    df['numstring'] = df['roundgzint'].astype(str)
    df['ld'] = df['numstring'].str[-1]
    df['ldan'] = df['ld'].astype(int)

    # Calculate rgzbotm based on conditions
    df['rgzbotm'] = np.where(df['GlobalZ_m'] > 0,
                            df['roundgzint'] - df['ldan'],
                            df['roundgzint'] + df['ldan'])
    
    condition1 = (df['GlobalZ_m'] >= -450) &(df['GlobalZ_m'] <= 300) & (df['ld'].isin(['8','9'])) & (df['GlobalZ_m'] > 0)

    # Assign values based on conditions
    df.loc[condition1, 'topelv1'] = df['nearest10int']
    df.loc[condition1, 'botelv1'] = df['topelv1'] - 5

    condition2 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ld'].isin(['8','9'])) & (df['GlobalZ_m'] < 0)

    df.loc[condition2, 'botelv1'] = df['nearest10int']
    df.loc[condition2, 'topelv1'] = df['botelv1'] + 5

    condition3 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ld'].isin(['1','2'])) & (df['GlobalZ_m'] > 0)

    df.loc[condition3, 'botelv1'] = df['nearest10int']
    df.loc[condition3, 'topelv1'] = df['botelv1'] + 5

    condition4 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ld'].isin(['1','2'])) & (df['GlobalZ_m'] < 0)

    df.loc[condition4, 'topelv1'] = df['nearest10int']
    df.loc[condition4, 'botelv1'] = df['topelv1'] - 5

    condition5 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ld'] == '0') & (df['GlobalZ_m'] < df['roundgzint']) & (df['GlobalZ_m'] > 0)

    df.loc[condition5, 'topelv1'] = df['nearest10int']
    df.loc[condition5, 'botelv1'] = df['topelv1'] - 5

    condition6 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ld'] == '0') & (df['GlobalZ_m'] < df['roundgzint']) & (df['GlobalZ_m'] < 0)

    df.loc[condition6, 'topelv1'] = df['nearest10int']
    df.loc[condition6, 'botelv1'] = df['topelv1'] - 5

    condition7 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ld'] == '0') & (df['GlobalZ_m'] > df['roundgzint']) & (df['GlobalZ_m'] > 0)

    df.loc[condition7, 'botelv1'] = df['nearest10int']
    df.loc[condition7, 'topelv1'] = df['botelv1'] + 5

    condition8 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ld'] == '0') & (df['GlobalZ_m'] > df['roundgzint']) & (df['GlobalZ_m'] < 0)

    df.loc[condition8, 'botelv1'] = df['nearest10int']
    df.loc[condition8, 'topelv1'] = df['botelv1'] + 5

    condition9=(df['GlobalZ_m']>=-450)&(df['GlobalZ_m'] <= 300)&(df['ld']=='0')&(df['GlobalZ_m']==df['roundgzint'])&(df['GlobalZ_m']>0)

    df.loc[condition9, 'topelv1'] = df['nearest10int']
    df.loc[condition9, 'botelv1'] = df['topelv1'] - 5
    df.loc[condition9, 'botelv2'] = df['nearest10int']
    df.loc[condition9, 'topelv2'] = df['botelv2'] + 5

    condition10 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ld'] == '0') & (df['GlobalZ_m'] == df['roundgzint']) & (df['GlobalZ_m'] < 0)

    df.loc[condition10, 'botelv1'] = df['nearest10int']
    df.loc[condition10, 'topelv1'] = df['botelv1'] + 5
    df.loc[condition10, 'topelv2'] = df['nearest10int']
    df.loc[condition10, 'botelv2'] = df['topelv2'] - 5

    condition11=(df['GlobalZ_m']>=-450)&(df['GlobalZ_m'] <= 300)&(df['ld'].isin(['6','7']))&(df['GlobalZ_m']>0)

    df.loc[condition11, 'topelv1'] = df['nearest10int']
    df.loc[condition11, 'botelv1'] = df['topelv1'] - 5

    condition12=(df['GlobalZ_m']>=-450)&(df['GlobalZ_m']<=300)&(df['ld'].isin(['6','7']))&(df['GlobalZ_m']<0)

    df.loc[condition12,'botelv1']=df['nearest10int']
    df.loc[condition12,'topelv1']=df['botelv1']+5

    condition13=(df['GlobalZ_m']>=-450)&(df['GlobalZ_m']<=300)&(df['ld'].isin(['3','4']))&(df['GlobalZ_m']>0)

    df.loc[condition13, 'botelv1'] = df['nearest10int']
    df.loc[condition13, 'topelv1'] = df['botelv1'] + 5

    condition14 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300)&(df['ld'].isin(['3','4']))&(df['GlobalZ_m'] < 0)

    df.loc[condition14, 'topelv1'] = df['nearest10int']
    df.loc[condition14, 'botelv1'] = df['topelv1'] - 5

    condition15=(df['GlobalZ_m']>=-450)&(df['GlobalZ_m']<=300)&(df['ldan']==5)&(df['GlobalZ_m']==df['roundgzint'])&(df['GlobalZ_m'] > 0)

    df.loc[condition15, 'botelv1'] = df['rgzbotm']
    df.loc[condition15, 'topelv1'] = df['botelv1'] + 5
    df.loc[condition15, 'botelv2'] = df['roundgz']
    df.loc[condition15, 'topelv1'] = df['botelv2'] + 5

    condition16 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ldan'] == 5) & (df['GlobalZ_m'] == df['roundgzint']) & (df['GlobalZ_m'] < 0)

    df.loc[condition16, 'botelv1'] = df['roundgzint']
    df.loc[condition16, 'topelv1'] = df['botelv1'] + 5
    df.loc[condition16, 'topelv2'] = df['roundgzint']
    df.loc[condition16, 'botelv2'] = df['topelv2'] - 5

    condition17 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300) & (df['ldan'] == 5) & (df['GlobalZ_m'] < df['roundgzint']) & (df['GlobalZ_m'] > 0)

    df.loc[condition17, 'topelv1'] = df['roundgzint']
    df.loc[condition17, 'botelv1'] = df['topelv1'] - 5

    condition18 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300)& (df['ldan'] == 5) & (df['GlobalZ_m'] < df['roundgzint']) & (df['GlobalZ_m'] < 0)

    df.loc[condition18, 'topelv1'] = df['roundgzint']
    df.loc[condition18, 'botelv1'] = df['topelv1'] - 5

    condition19 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300)& (df['ldan'] == 5) & (df['GlobalZ_m'] > df['roundgzint']) & (df['GlobalZ_m'] > 0)

    df.loc[condition19, 'botelv1'] = df['roundgzint']
    df.loc[condition19, 'topelv1'] = df['botelv1'] + 5

    condition20 = (df['GlobalZ_m'] >= -450)&(df['GlobalZ_m'] <= 300)& (df['ldan'] == 5) & (df['GlobalZ_m'] > df['roundgzint']) & (df['GlobalZ_m'] < 0)

    df.loc[condition20, 'botelv1'] = df['roundgzint']
    df.loc[condition20, 'topelv1'] = df['botelv1'] + 5
    
    df['botelv1int'] = df['botelv1'].astype(int)
    print('botelv1int:',df['botelv1int'][0])
    df['botelv1round'] = df['botelv1'].round()
    print('botelv1round:',df['botelv1round'][0])
    df['botelv1roundn1'] = df['botelv1'].round(-1)
    print('botelv1roundn1:',df['botelv1roundn1'][0])
    df['botelv1round0'] = df['botelv1'].round(0)
    print('botelv1round0:',df['botelv1round0'][0])
    df['botelv1round1'] = df['botelv1'].round(1)
    print('botelv1round1:',df['botelv1round1'][0])
    df['botelv1str'] = df['botelv1'].astype(str)
    print('botelv1str:',df['botelv1str'][0])
    df['botelv1strrl2'] = df['botelv1str'].str[:-2]
    print('botelv1strrl2:',df['botelv1strrl2'][0])
    #sys.exit()
    ########################################
    df['topelv1int'] = df['topelv1'].astype(int)
    #df['botelv1']=int(df['botelv1'])
    #df['topelv1']=int(df['topelv1'])

    #df.loc[(df['botelv1']!=-9999)&(df['topelv1']!=-9999),'elevationSlice1'] = 'elv_res_' + df['topelv1'].astype(str) + '_' + df['botelv1'].astype(str) + 'm.tif'
    #df.loc[(df['botelv1']!=-9999)&(df['topelv1']!=-9999),'elevationSlice1'] = 'elv_res_' + df['topelv1int'] + '_' + df['botelv1int'] + 'm.tif'
    df.loc[(df['botelv1']!=-9999)&(df['topelv1']!=-9999),'elevationSlice1'] = 'elv_res_' + df['topelv1int'].astype(str) + '_' + df['botelv1int'].astype(str) + 'm.tif'
    #df['elevationSlice1'] = 'elv_res_' + df['topelv1'].astype(str) + '_' + df['botelv1'].astype(str) + 'm.tif'

    #df['elevationSlice2'] = 'elv_res_' + df['topelv2'].astype(str) + '_' + df['botelv2'].astype(str) + 'm.tif'
    
    for index, row in df.iterrows():
        if row['elevationSlice1']!='xx':
            tiffName1 = row['elevationSlice1']
            tiffName2 = row['elevationSlice2']

            #with rasterio.open(os.path.join(r"C:\Users\chaugh\Documents\micheal_g_files\MY_TIFFS\MY_TIFFS", tiffName1)) as src1:
            #Looks like src2 is not used in the code...
            # if (tiffName2!=- 'xx'):
            #     src2 = rasterio.open(os.path.join...)
            with rasterio.open(os.path.join(r"C:\github\AEM\examples\OneDrive_1_2-24-2022\MY_TIFFS", tiffName1)) as src1:
                raster_value = next(src1.sample([(row['x_4326'], row['y_4326'])]))[0]
                raster_value2 = next(src1.sample([(row['x_5070'], row['y_5070'])]))[0]
                #print('raster_value:',raster_value)
                #print('raster_value2:',raster_value2)
                #sys.exit()
                #print("Raster Value: ", raster_value)
                df.at[index, 'Raster Value'] = raster_value

            df.loc[df['Raster Value']>(10**30),'Raster Value']=-9999
            df['res_ohm_m1'] = df['res_ohm_m1'].astype(float)
            df.at[index, 'res_ohm_m1'] = df['Raster Value'][index]
        
    df.loc[df['res_ohm_m1']==-9999,'res_ohm_m1']=np.nan
    #df['log10res']=log10(df['res_ohm_m1'])
    df['log10res'] = np.where(df['res_ohm_m1'] > 0, np.log10(df['res_ohm_m1']), np.nan)
    #df.to_csv("temp_file.csv")
    print("Extraction complete for file: ", filename)
    print("Thank You!")
    #sys.exit()
    #coords=[]
    PIDs=[]
    #     latints=[]
    #     longints=[]
    #     elvints=[]
    #     for index,row in df.iterrows():
    #         coords=[]
    #         #latints=[]
    #         #longints=[]
    #         #elvints=[]
    #         coords.append(row['x_4326'])
    #         coords.append(row['y_4326'])
    #         coords.append(row['GlobalZ_m'])
    #         PT=np.array(coords)
    #         onelocrec=ds.sel(z=PT[2],y=PT[1],x=PT[0],method='nearest')
    #         latint=onelocrec.y_bnds
    #         longint=onelocrec.x_bnds
    #         elvint=onelocrec.z_bnds
    #         row['latintbot']=latint[0]
    #         row['longintbot']=longint[0]
    #         row['elvintbot']=elvint[0]
    #         row['latinttop']=latint[1]
    #         row['longinttop']=longint[1]
    #         row['elvinttop']=elvint[1]
    #         PID=row['Particle ID']
    #         PIDs.append(PID)
    #         latints.append(latint)
    #         longints.append(longint)
    #         elvints.append(elvint)
    #d={'PID':PIDs,'latint':latints,'longint':longints,'elvint':elvints}
    #medianmeanagesdf=pd.DataFrame(data=d)
    #sys.exit()

    #Only change I made from here on was to the handling of sitenos and the construction of the well_summ_parts dataframe.
    #     meanres=np.nanmean(df['res_ohm_m1'])  # arithmetic average of all resistivities for all particles tracked from the well.
    #     df.loc[:,'meanres']=meanres
    #     meanlog10res=np.nanmean(df['log10res'])
    #     df.loc[:,'meanlog10res']=meanlog10res

    #     # Goodness of fit test to see if log resistivities are normally distributed in which case the overall distribution is lognormal. Here, this is being applied to all particles tracked from the well.
    #     loc,scale = np.nanmean(df['log10res']), np.std(df['log10res'], ddof=1)
    #     cdf = stats.norm(loc, scale).cdf
    #     res=stats.ks_1samp(df['log10res'], cdf)
    #     ###print(res)
    #     if res.pvalue<.05:
    #         print('not log normal')
    #         df.loc[:,'logNormality']='not log normal'
    #     else:
    #         print('log normal')
    #         df.loc[:,'logNormality']='log normal'
	
    #######################
    # Now group df by PID and calculate	variance and effective resistivity for each particle. Then calculate average of variances and variance of effective resistivities for each well. [Each name is a PID (particle). Each group is the df for that particle. Numrecs is the number of locations recorded along the particle's path. Each q is a location recorded along the particle's path.]

    PIDgrp=df.groupby('Particle ID')
    #Flow path- and Well-level lists
    #     sitenos=[]
    #     sites=[]
    number_of_locs=[]
    pathlengths=[]
    RESeffs=[]#before looping to the next particle, store this particle's RESeff in the list RESeffs.
    VRs=[]
    cvrs=[]
    #numslocs=[]
    #numsparts=[]
    #     MERs=[]
    #     VERs=[]
    #     AVRs=[]
    #     VVRs=[]
    #     cvers=[]
    #     acvrs=[]
    #     cvcvrs=[]
    DFMERs=[]
    DFMSERs=[]
    DFMVRs=[]
    DFMSVRs=[]
    DFMcVRs=[]
    DFMScVRs=[]
    logNormality=[]
    #     snRESeffs=[]
    #     logRESeffs=[]
    #     snlogRESeffs=[]
    #     logVARs=[]#before looping to the next particle, store this particle's variance of log resistivities along its flowpath in the list logVARs.
    #     VARs_sn=[]
    #     VARs_snlogres=[]

    for name,group in PIDgrp:#for each particle:
        print('Name:',name)
        #         grpshp=group.shape
        #         numlocs=grpshp[0]  # numlocs = the number of locations recorded along the particle's flowpath.
        #         nm1=numlocs-1
        #         nm2=numlocs-2
        #         INDX=range(numlocs)
        
        ##SORT FIRST##
        #grpdf=group.drop_duplicates(subset=['Particle ID','Global X','Global Y','Global Z'])
        #Cumulative time step is the time step of the MODFLOW simulation corresponding to a given row of a pathline
        #file. It starts at 1 for the first time step of the MODFLOW simulation and increments sequentially through
        #the last time step of the MODFLOW simulation. Tracking Times is the value of modpath time corresponding to
        #the particle at the specified location. For AEM MPK calcs, sort particle's location records so that they are in order from the
        #first time step of the MODFLOW simulation to the time step of the MODFLOW simulation when the particle is at
        #the well, i.e., from least to greatest cumulative time step and from greatest to least tracking time.
        #group = grpdf.sort_values(by=['Cumulative Time Step','Tracking Time'],ascending=[False,True])
        #For AEM MPK calcs
        group = group.sort_values(by=['Cumulative Time Step','Tracking Time'],ascending=[True,False])
        #group.to_csv(f'particle_{name}_v2.csv', index=False)
        grpdf = group.reset_index(drop=True)
        grpshp=grpdf.shape
        numlocs=grpshp[0]  # numlocs = the number of locations recorded along the particle's flowpath.
        nm1=numlocs-1
        nm2=numlocs-2
        print('nm2:',nm2)
        r=np.linspace(1,nm1,nm1)
        #grpdf=pd.DataFrame(data=group, index=INDX)
        #grpdf.to_csv(f'grpdf_for_{name}.csv', index=False)
        #print('res_ohm_m1: ', grpdf['res_ohm_m1'])
        #sitenolist=grpdf['siteno'].tolist()
        sitenos=grpdf['siteno'].tolist()
        sites=grpdf['pids'].tolist()
        lx=grpdf['Local X'].tolist()
        ly=grpdf['Local Y'].tolist()
        lz=grpdf['Local Z'].tolist()
        gx=grpdf['Global X'].tolist()
        print('len(gx):',len(gx))
        gy=grpdf['Global Y'].tolist()
        gz=grpdf['Global Z'].tolist()
        th=grpdf['thickness'].tolist()
        #         dt=grpdf['distance traveled'].tolist()
        #         cdti=grpdf['CumulativeDistanceTraveledIndex'].tolist()
        #         cdt=grpdf['CumulativeDistanceTraveled'].tolist()
        #         dtwc=grpdf['distance traveled within cell'].tolist()
        #         cdtwci=grpdf['CumulativeDistanceTraveledwinCellIndex'].tolist()
        #         cdtwc=grpdf['CumulativeDistanceTraveledwinCell'].tolist()
        #         ci=grpdf['cellIndex'].tolist()
        layer=grpdf['Layer'].tolist()
        row=grpdf['Row'].tolist()
        column=grpdf['Column'].tolist()
        mpk=grpdf['mpk'].tolist()
        #         mplk=grpdf['mplogk'].tolist()
        #         mpsnk=grpdf['mpsnk'].tolist()
        #         mpsnlk=grpdf['mpsnlogk'].tolist()
        rom1=grpdf['res_ohm_m1'].tolist()
        ltr=grpdf['log10res'].tolist()
        #         DFMER=grpdf['erdfm'].tolist()
        #         DFMSER=grpdf['erdfms'].tolist()
        DFMRs=grpdf['dfmr'].tolist()
        DFMSRs=grpdf['dfmsr'].tolist()
        #         laintbs=grpdf['latintbot'].tolist()
        #         lointbs=grpdf['longintbot'].tolist()
        #         elintts=grpdf['elvinttop'].tolist()
        #         laintts=grpdf['latinttop'].tolist()
        #         lointts=grpdf['longinttop'].tolist()
        #         elintbs=grpdf['elvintbot'].tolist()
        #         DFMERs=grpdf['dfmer'].tolist()
        #         DFMSERs=grpdf['dfmser'].tolist()
        #         DFMVRs=grpdf['dfmvr'].tolist()
        #         DFMSVRs=grpdf['dfmvr'].tolist()
        #         DFMcVRs=grpdf['dfmcvr'].tolist()
        #         DFMScVRs=grpdf['dfmscvr'].tolist()
        #rsn=grpdf['ressn'].tolist()
        #lrsn=grpdf['logressn'].tolist()

        #dt[0]=0
        if numlocs<2:
            sitenos[0]=siteno
            sites[0]=name
            th[0]=-9999
            mpk[0]=-9999
            #             sitenos[nm1]=siteno
            #             sites[nm1]=name
            #             th[nm1]=.5*((((gx[nm1]-gx[nm2])**2)+((gy[nm1]-gy[nm2])**2)+((gz[nm1]-gz[nm2])**2))**.5)
            #             mpk[nm1]=th[nm1]/rom1[nm1]
            #         cdti[0]=0
            #         cdt[0]=0
            #         dtwc[0]=0
            #         cdtwci[0]=0
            #         cdtwc[0]=0
            #         ci[0]=0
            print('numlocs:',numlocs)#number of particle locations recorded for one particle.

            # This third part of the script is for calculating distances between sequential recorded particle locations.
            #numlocs=1
        
            RESeffs.append(-9999)
            VRs.append(-9999)
            cvrs.append(-9999)
            #numslocs.append(-9999)
            #numsparts.append(-9999)
            #             MERs.append(-9999)
            #             VERs.append(-9999)
            #             AVRs.append(-9999)
            #             VVRs.append(-9999)
            #             cvers.append(-9999)
            #             acvrs.append(-9999)
            #             cvcvrs.append(-9999)
            DFMERs.append(-9999)
            DFMSERs.append(-9999)
            DFMVRs.append(-9999)
            DFMSVRs.append(-9999)
            DFMcVRs.append(-9999)
            DFMScVRs.append(-9999)
            #logNormality=.append(0)
            #print('RESeffs:',RESeffs)
            #sys.exit()
            #####################################################################################
            #START OF AEM MPK CALCS
            #####################################################################################
            #         else:
            #             sitenos[0]=siteno
            #             sites[0]=name
            #             th[0]=0
            #             mpk[0]=0
            #             for q in r:
            #                 sitenos[q]=siteno
            #                 sites[q]=name
            #                 if rom1[q-1]==rom1[q]:
            #                     th[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**.5
            #                     mpk[q]=th[q]/rom1[q]
            #                 else:
            #                     if latint[q]!=latint[q-1]:
            #                         if latinttop[q-1]==latintbot[q]:
            #                 sys.exit()
            #####################################################################################
            #END OF AEM MPK CALCS
            #####################################################################################
        else:
            sitenos[0]=siteno
            sites[0]=name
            th[0]=.5*((((gx[1]-gx[0])**2)+((gy[1]-gy[0])**2)+((gz[1]-gz[0])**2))**.5)
            mpk[0]=th[0]/rom1[0]
            sitenos[nm1]=siteno
            sites[nm1]=name
            th[nm1]=.5*((((gx[nm1]-gx[nm2])**2)+((gy[nm1]-gy[nm2])**2)+((gz[nm1]-gz[nm2])**2))**.5)
            mpk[nm1]=th[nm1]/rom1[nm1]
            #         cdti[0]=0
            #         cdt[0]=0
            #         dtwc[0]=0
            #         cdtwci[0]=0
            #         cdtwc[0]=0
            #         ci[0]=0
            print('numlocs:',numlocs)#number of particle locations recorded for one particle.
            r=np.linspace(1,nm2,nm2).astype(int)
            for q in r:#for each location recorded for the particle in question:
                th[q]=.5*((((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**.5)+.5*((((gx[q+1]-gx[q])**2)+((gy[q+1]-gy[q])**2)+((gz[q+1]-gz[q])**2))**.5)
                #dt[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**.5
                #cdti[q]=cdti[q-1]+1
                #cdt[q]=dt[q]+cdt[q-1]
                #########################################################
                #if (layer[q]==layer[q-1])&(row[q]==row[q-1])&(column[q]==column[q-1]):
                #    #dtwc[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**(.5)
                #    #cdtwci[q]=cdtwci[q-1]+1
                #    #cdtwc[q]=dtwc[q]+cdtwc[q-1]
                #    #ci[q]=ci[q-1]
                #else:
                #    dtwc[q]=0
                #    #cdtwci[q]=0
                #    #cdtwc[q]=0
                #    #ci[q]=ci[q-1]+1
                mpk[q]=(th[q])/(rom1[q])
                #mplk[q]=(dtwc[q])/(ltr[q])
                #mpsnk[q]=(dtwc[q])/(rsn[q])
                #mpsnlk[q]=(dtwc[q])/(lrsn[q])
                #print('Distances for particle {} calculated!'.format(name))
                sites[q]=name
                sitenos[q]=siteno
            #             print('thickness:',th)
            #             print('rom1:',rom1)
            #             print('mpk:',mpk)
            #d={'sitenumber':sitenos,'ParticleID':sites,'thickness':th,'rom1':rom1,'mpk':mpk}
            #well_summ_parts=pd.DataFrame(data=d)
            #well_summ_parts.loc[:,'avglogRES_well']=meanlog10res
            #well_summ_parts.to_csv(os.path.join('welSumms','well_summ_parts_{}_11451619check.csv'.format(siteno)))
            #sys.exit()
            #mpk=[np.nan if x==0 else x for x in mpk]
            RESeffdenominator=np.nansum(mpk)
            RESeffnumerator=np.nansum(th)
            RESeff=RESeffnumerator/RESeffdenominator
            grpshp=grpdf.shape
            numlocs=grpshp[0]
            print('numlocs:',numlocs)
            r=len(rom1)
            print('len(rom1):',r)
            #r=range(numlocs)
            r=range(r)
            for q in r:
                if rom1[q]!=0:
                    DFMRs[q]=rom1[q]-RESeff
                    DFMSRs[q]=(DFMRs[q])**2
                else:
                    DFMRs[q]=-9999
                    DFMSRs[q]=-9999
            DFMSRs=[np.nan if x==-9999 else x for x in DFMSRs]
            #DFMSRs=DFMSRs.replace(-9999,np.nan)
            VRnumerator=np.nansum(DFMSRs)
            DFMSRs=[x for x in DFMSRs if not math.isnan(x)]
            length=len(DFMSRs)
            print('length:',length)
            print('numlocs:',numlocs)
            #sys.exit()
            VRdenominator=length-1
            ######################
            #sys.exit()
            if length>1:
                VR=VRnumerator/VRdenominator
            else:
                VR=-9999
            print('VR:',VR)
            VRs.append(VR)
            if VR!=-9999:
                cvr=((VR)**.5)/(RESeff)
            else:
                cvr=-9999
            print('cvr:',cvr)
            cvrs.append(cvr)
            ##################
            ######################
            #VR=VRnumerator/VRdenominator
            #cvr=((VR)**.5)/(RESeff)
            #             cvrs.append(cvr)
            #             print('cvrs:',cvrs)
            #print('VRs:',VRs)
            VRs.append(VR)
            #VRs=[np.nan if x==-9999 else x for x in VRs]
            if RESeffdenominator!=0:
                RESeffs.append(RESeff)
            else:
                RESeffs.append(-9999)
            print('RESeff:',RESeff)
            #             DFMER=RESeff-MER
            #             DFMERs.append(DFMER)
            #             DFMSER=(DFMER)**2
            #             DFMSERs.append(DFMSER)
            #             DFMVR=VR-AVR
            #             DFMVRs.append(DFMVR)
            #             DFMSVR=(DFMVR)**2
            #             DFMSVRs.append(DFMSVR)
            #             DFMcVR=cvr-acvr
            #             DFMcVRs.append(DFMcVR)
            #             DFMScVRs[q]=(DFMcVRs[q])**2
            #             DFMScVRs.append(DFMScVR)
    #VRs=[np.nan if x==-9999 else x for x in VRs]
    #cvrs=[np.nan if x==-9999 else x for x in cvrs]
    #RESeffs=RESeffs.replace(-9999,np.nan)
    RESeffs=[np.nan if x==-9999 else x for x in RESeffs]
    MER=np.nanmean(RESeffs)
    print('MER:',MER)
    MERs.append(MER)
    #sys.exit()
    ##########################
    r=len(RESeffs)
    print('r:',r)
    r=range(r)
    for q in r:
        #print('RESeff index:',r)
        DFMER=RESeffs[q]-MER
        DFMERs.append(DFMER)
        DFMSER=(DFMER)**2
        DFMSERs.append(DFMSER)
    ##########################
    DFMSERs=[np.nan if x==-9999 else x for x in DFMSERs]
    #DFMSER=DFMSER.replace(-9999,np.nan)
    VERnumerator=np.nansum(DFMSERs)
    DFMSERs=[x for x in DFMSERs if not math.isnan(x)]
    length=len(DFMSERs)
    print('lenDFMSERs:',length)
    print('numlocs:',numlocs)
    #sys.exit()
    VERdenominator=length-1
    if length>1:
        VER=VERnumerator/VERdenominator
    else:
        VER=-9999
    print('VER:',VER)
    VERs.append(VER)
    if VER!=-9999:
        cver=((VER)**.5)/(MER)
    else:
        cver=-9999
    print('cver:',cver)
    cvers.append(cver)
    ##################
    VRs=[np.nan if x==-9999 else x for x in VRs]
    #VRs=VRs.replace(-9999,np.nan)
    AVR=np.nanmean(VRs)    
    print('AVR:',AVR)
    AVRs.append(AVR)
    cvrs=[np.nan if x==-9999 else x for x in cvrs]
    #cvrs=cvrs.replace(-9999,np.nan)
    acvr=np.nanmean(cvrs)
    print('acvr:',acvr)
    acvrs.append(acvr)
    ##########################
    r=len(VRs)
    print('r:',r)
    r=range(r)
    for q in r:
        #print('RESeff index:',r)
        DFMVR=VRs[q]-AVR
        DFMVRs.append(DFMVR)
        DFMSVR=(DFMVR)**2
        DFMSVRs.append(DFMSVR)
    ##########################
    DFMSVRs=[np.nan if x==-9999 else x for x in DFMSVRs]
    #DFMSVR=DFMSVR.replace(-9999,np.nan)
    VVRnumerator=np.nansum(DFMSVRs)
    DFMSVRs=[x for x in DFMSVRs if not math.isnan(x)]
    length=len(DFMSVRs)
    print('length:',length)
    print('numlocs:',numparts)
    VVRdenominator=length-1
    if length>1:
        VVR=VVRnumerator/VVRdenominator
    else:
        VVR=-9999
    print('VVR:',VVR)
    VVRs.append(VVR)
    #     if VVR!=-9999
    #         cvVr=((VVR)**.5)/(MER)
    #     else:
    #         cvVr=-9999
    #     print('cvVr:',cvVr)
    #     cvVrs.append(cvVr)
    ##################
    #     VVR=VVRnumerator/VVRdenominator
    #     print('VVR:',VVR)
    #     VVRs.append(VVR)
    ##################
    #     r=len(cvrs)
    #     print('r:',r)
    #     r=range(r)
    #     for q in r:
    #         DFMcVRs[q]=cvrs[q]-acvr
    #         DFMScVRs[q]=(DFMcVRs[q])**2
    ##########################
    r=len(cvrs)
    print('r:',r)
    r=range(r)
    for q in r:
        #print('RESeff index:',q)
        DFMcVR=cvrs[q]-acvr
        DFMcVRs.append(DFMcVR)
        DFMScVR=(DFMcVR)**2
        DFMScVRs.append(DFMScVR)
    ##########################
    DFMScVRs=[np.nan if x==-9999 else x for x in DFMScVRs]
    #DFMScVR=DFMScVR.replace(-9999,np.nan)
    VcVRnumerator=np.nansum(DFMScVR)
    DFMScVRs=[x for x in DFMScVRs if not math.isnan(x)]
    length=len(DFMScVRs)
    print('length:',length)
    print('numlocs:',numparts)
    VcVRdenominator=length-1
    ##########
    if length>1:
        VcVR=VcVRnumerator/VcVRdenominator
    else:
        VcVR=-9999
    print('VcVR:',VcVR)
    VcVRs.append(VcVR)
    if VcVR!=-9999:
        cvcvr=((VcVR)**.5)/acvr
    else:
        cvcvr=-9999
    print('cvcvr:',cvcvr)
    cvcvrs.append(cvcvr)
    ##################
    ##########
    #     VcVR=VcVRnumerator/VcVRdenominator
    #     print('VcVR:',VcVR)
    #     VcVRs.append(VcVR)
    #     cvcvr=((VcVR)**.5)/acvr
    #     print('cvcvr:',cvcvr)
    #     cvcvrs.append(cvcvr)
    #sys.exit()
    VERs=[np.nan if x==-9999 else x for x in VERs]
    d={'sitenumber':sitenolist,'numparts':numsparts,'MER':MERs,'VER':VERs,'AVR':AVRs,'VVR':VVRs,'cver':cvers,'acvr':acvrs,'cvcvr':cvcvrs}
    well_summ_parts=pd.DataFrame(data=d)
    #well_summ_parts.loc[:,'avglogRES_well']=meanlog10res
    well_summ_parts.to_csv(os.path.join('welSumms','well_summ_parts_{}_3205.csv'.format(siteno)))
    #print('# file finished: ', idx + 1)
    print(f"File: {filename} has finished!")
    print('Grazie!')
    #sys.exit()
    print("Thank you!")

print("All files have finished processing!")
sys.exit()
###############################################################################################################
###############################################################################################################
###############################################################################################################
for x in sitenolist:#print('Distances for all particles calculated!')
    cumDisPart=max(cdt)
    if numlocs>1:
        if numlocs>=2:
            r=np.linspace(1,numlocs-1,numlocs-1).astype(int)
            for q in r:
                if (lx[q]!=lx[q-1])|(ly[q]!=ly[q-1])|(lz[q]!=lz[q-1]):

                    dt[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**.5
                    cdti[q]=cdti[q-1]+1
                    cdt[q]=dt[q]+cdt[q-1]
                    #########################################################
                    if (layer[q]==layer[q-1])&(row[q]==row[q-1])&(column[q]==column[q-1]):
                        dtwc[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**(.5)
                        cdtwci[q]=cdtwci[q-1]+1
                        cdtwc[q]=dtwc[q]+cdtwc[q-1]
                        ci[q]=ci[q-1]

                    else:
                        dtwc[q]=0
                        cdtwci[q]=0
                        cdtwc[q]=0
                        ci[q]=ci[q-1]+1
                    mpk[q]=(dtwc[q])/(rom1[q])
                    mplk[q]=(dtwc[q])/(ltr[q])
                    mpsnk[q]=(dtwc[q])/(rsn[q])
                    mpsnlk[q]=(dtwc[q])/(lrsn[q])
                    #print('Distances for particle {} calculated!'.format(name))
            #print('Distances for all particles calculated!')
            cumDisPart=max(cdt)
            
            #RESeffdenom=math.fsum(mpk)
            RESeffdenom=np.nansum(mpk)
            
            if RESeffdenom!=0:
                RESeff=(max(cdt))/RESeffdenom  # effective resistivity of one particle's flowpath.
            else:
                print('else condition met for RESeff')
                RESeff=np.nan

            #logRESeffdenom=math.fsum(mplk)
            logRESeffdenom=np.nansum(mplk)
            if logRESeffdenom!=0:
                logRESeff=(max(cdt))/logRESeffdenom  # effective resistivity of one particle's flowpath.
            else:
                print('else condition met for logRESeff')
                logRESeff=np.nan
        ############################################################################################
        
        #smeanres=np.nanmean(grpdf['res_ohm_m1'])  # Average resistivity recorded along a particle's path
        #print("smeanres: ", smeanres)
        smeanlogres=np.nanmean(grpdf['log10res'])  # Average log resistivity recorded along a particle's path
        #print('smeanlogres: ', smeanlogres)
        #One thing I've noticed: there is often only 1 value in res_ohm_m1, so the mean of that column is equal to the values, so in the line below, res_ohm_m1 - smeanres = 0, hence why
        grpdf['dfms']=(grpdf['res_ohm_m1']-smeanres)**2
        # dfms_df = grpdf[['dfms']]

        # # Save the dfms DataFrame to a CSV file
        # dfms_df.to_csv(f'dfms_column_{name}.csv', index=False)

        grpdf['dfms_logres']=(grpdf['log10res']-smeanlogres)**2
        #print('grpdf[dfms_logres]: ', grpdf['dfms_logres'])
        #grpdf.to_csv(f'grpdf_v2_{name}.csv', index=False)
        # Calculate standard normal resistivity
        #VAR=(math.fsum(grpdf['dfms']))/(numlocs-1)  # Variance of raw resistivities along one particle's flowpath.
        VAR = (np.nansum(grpdf['dfms']))/(numlocs - 1)
        #print("sum: ", np.nansum(grpdf['dfms']))
        #print("Numlocs: ", numlocs)
        #print('VAR: ', VAR)
        stdres=VAR**.5
        grpdf['ressn']=(grpdf['res_ohm_m1']-smeanres)/stdres
        ###########################++++++++++++++++++++++++++++++
        #VARlogres=(math.fsum(grpdf['dfms_logres']))/(numlocs-1)  # Variance of log resistivities along one particle's flowpath.
        VARlogres = (np.nansum(grpdf['dfms_logres'])) / (numlocs - 1)
        #print('VARlogres: ', VARlogres)
        stdlogres=(VARlogres)**.5
        grpdf['logressn']=(grpdf['log10res']-smeanlogres)/stdlogres

        sitenolist=grpdf['siteno'].tolist()
        lx=grpdf['Local X'].tolist()
        ly=grpdf['Local Y'].tolist()
        lz=grpdf['Local Z'].tolist()
        gx=grpdf['Global X'].tolist()
        gy=grpdf['Global Y'].tolist()
        gz=grpdf['Global Z'].tolist()
        #         dt=grpdf['distance traveled'].tolist()
        #         cdti=grpdf['CumulativeDistanceTraveledIndex'].tolist()
        #         cdt=grpdf['CumulativeDistanceTraveled'].tolist()
        #         dtwc=grpdf['distance traveled within cell'].tolist()
        #         cdtwci=grpdf['CumulativeDistanceTraveledwinCellIndex'].tolist()
        #         cdtwc=grpdf['CumulativeDistanceTraveledwinCell'].tolist()
        #         ci=grpdf['cellIndex'].tolist()
        layer=grpdf['Layer'].tolist()
        row=grpdf['Row'].tolist()
        column=grpdf['Column'].tolist()
        mpk=grpdf['mpk'].tolist()
        #         mplk=grpdf['mplogk'].tolist()
        #         mpsnk=grpdf['mpsnk'].tolist()
        #         mpsnlk=grpdf['mpsnlogk'].tolist()
        rom1=grpdf['res_ohm_m1'].tolist()
        ltr=grpdf['log10res'].tolist()
        rsn=grpdf['ressn'].tolist()
        lrsn=grpdf['logressn'].tolist()

        dt[0]=0
        cdti[0]=0
        cdt[0]=0
        dtwc[0]=0
        cdtwci[0]=0
        cdtwc[0]=0
        ci[0]=0
        print('numlocs:',numlocs)

        # This third part of the script is for calculating distances between sequential recorded particle locations.
        if numlocs>=2:
            r=np.linspace(1,numlocs-1,numlocs-1).astype(int)
            for q in r:
                dt[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**.5
                cdti[q]=cdti[q-1]+1
                cdt[q]=dt[q]+cdt[q-1]
                #########################################################
                if (layer[q]==layer[q-1])&(row[q]==row[q-1])&(column[q]==column[q-1]):
                    dtwc[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**(.5)
                    cdtwci[q]=cdtwci[q-1]+1
                    cdtwc[q]=dtwc[q]+cdtwc[q-1]
                    ci[q]=ci[q-1]
                else:
                    dtwc[q]=0
                    cdtwci[q]=0
                    cdtwc[q]=0
                    ci[q]=ci[q-1]+1
                mpk[q]=(dtwc[q])/(rom1[q])
                mplk[q]=(dtwc[q])/(ltr[q])
                mpsnk[q]=(dtwc[q])/(rsn[q])
                mpsnlk[q]=(dtwc[q])/(lrsn[q])
                #print('Distances for particle {} calculated!'.format(name))
            #print('Distances for all particles calculated!')
            cumDisPart=max(cdt)
        if numlocs>=2:
            r=np.linspace(1,numlocs-1,numlocs-1).astype(int)
            for q in r:
                if (lx[q]!=lx[q-1])|(ly[q]!=ly[q-1])|(lz[q]!=lz[q-1]):

                    dt[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**.5
                    cdti[q]=cdti[q-1]+1
                    cdt[q]=dt[q]+cdt[q-1]
                    #########################################################
                    if (layer[q]==layer[q-1])&(row[q]==row[q-1])&(column[q]==column[q-1]):
                        dtwc[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**(.5)
                        cdtwci[q]=cdtwci[q-1]+1
                        cdtwc[q]=dtwc[q]+cdtwc[q-1]
                        ci[q]=ci[q-1]

                    else:
                        dtwc[q]=0
                        cdtwci[q]=0
                        cdtwc[q]=0
                        ci[q]=ci[q-1]+1
                    mpk[q]=(dtwc[q])/(rom1[q])
                    mplk[q]=(dtwc[q])/(ltr[q])
                    mpsnk[q]=(dtwc[q])/(rsn[q])
                    mpsnlk[q]=(dtwc[q])/(lrsn[q])
                    #print('Distances for particle {} calculated!'.format(name))
            #print('Distances for all particles calculated!')
            cumDisPart=max(cdt)
            
            #RESeffdenom=math.fsum(mpk)
            RESeffdenom=np.nansum(mpk)
            
            if RESeffdenom!=0:
                RESeff=(max(cdt))/RESeffdenom  # effective resistivity of one particle's flowpath.
            else:
                print('else condition met for RESeff')
                RESeff=np.nan

            #logRESeffdenom=math.fsum(mplk)
            logRESeffdenom=np.nansum(mplk)
            if logRESeffdenom!=0:
                logRESeff=(max(cdt))/logRESeffdenom  # effective resistivity of one particle's flowpath.
            else:
                print('else condition met for logRESeff')
                logRESeff=np.nan

            ###########################++++++++++++++++++++++++++++++
            #calculate variance of standard normal resistivities along this particle's flowpath.
            grpdf=grpdf.join(pd.DataFrame({'ressn2':rsn}))
            smeansnres=np.nanmean(rsn)
            grpdf['dfms_snres']=(grpdf['ressn2']-smeansnres)**2
            #VARsnres=(math.fsum(grpdf['dfms_snres']))/(numlocs-1)
            VARsnres = (np.nansum(grpdf['dfms_snres']))/(numlocs - 1)
            stdsnres=(VARsnres)**.5
            #print('VARsnres: ', VARsnres)
            ###########################++++++++++++++++++++++++++++++
            #calculate variance of standard normal log resistivities along this particle's flowpath.
            grpdf=grpdf.join(pd.DataFrame({'lrsn2':lrsn}))
            smeansnlogres=np.nanmean(grpdf['lrsn2'])
            grpdf['dfms_logsnres']=(grpdf['lrsn2']-smeansnlogres)**2
            #VARsnlogres=(math.fsum(grpdf['dfms_logsnres']))/(numlocs-1)
            VARsnlogres=(np.nansum(grpdf['dfms_logsnres']))/(numlocs-1)
            stdsnlogres=(VARsnlogres)**.5
            #print('VARsnlogres: ', VARsnlogres)

            #calculate effective res of standard normal resistivities along this particle's flowpath.
            grpdf=grpdf.join(pd.DataFrame({'mpsnk2':mpsnk,'cdt2':cdt}))
            #snRESeffdenom=math.fsum(grpdf['mpsnk2']) 
            snRESeffdenom=np.nansum(grpdf['mpsnk2']) 
            if snRESeffdenom!=0:
                snRESeff=(max(grpdf['cdt2']))/snRESeffdenom#effective resistivity of one particle's flowpath.
            else:
                print('else condition met for snRESeff')
                snRESeff=np.nan

            #calculate effective res of standard normal log resistivities along this particle's flowpath.
            grpdf=grpdf.join(pd.DataFrame({'mpsnlk2':mpsnlk}))
            #snlogRESeffdenom=math.fsum(grpdf['mpsnlk2'])
            snlogRESeffdenom=np.nansum(grpdf['mpsnlk2'])
            if snlogRESeffdenom!=0:
                snlogRESeff=(max(grpdf['cdt2']))/snlogRESeffdenom#effective resistivity of one particle's flowpath.
            else:
                print('else condition met for snlogRESeff')
                snlogRESeff=np.nan
            
            pathlengths.append(cumDisPart)
            RESeffs.append(RESeff)#before looping to the next particle, store this particle's RESeff in the list RESeffs.
            snRESeffs.append(snRESeff)
            logRESeffs.append(logRESeff)
            snlogRESeffs.append(snlogRESeff)
            VARs.append(VAR)#before looping to the next particle, store the variance of the resistivities along this particle's flowpath in the list VARs.
            logVARs.append(VARlogres)#before looping to the next particle, store this particle's variance of log resistivities along its flowpath in the list logVARs.
            VARs_sn.append(VARsnres)
            VARs_snlogres.append(VARsnlogres)
            sites.append(name)#before looping to the next particle, store this particle's ID in the list sites.
            number_of_locs.append(numlocs)
            print('Calculations for particle {} completed!'.format(name))
            #print("VARs: ", VARs)
    print('Calculations for all particles completed!')

    #I added thes three lines to get around an error I was getting (length of the lists was not the same for creating the data frame 'well_summ_parts'), 
    #you'll want to double check that this is treating the 'siteno's the right way
    #these three lines take the 'siteno' list that is created from the grpdf and filters out all NA values, then repeats the non-NA siteno for the length of the other lists used to create the dataframe
    siteno_series = pd.Series(sitenolist)
    # Find the first non-NaN value
    first_non_na = siteno_series.dropna().iloc[0] if not siteno_series.dropna().empty else None
    # Repeat the first non-NaN value 100 times
    new_sitenos = [first_non_na] * len(sites) if first_non_na is not None else []
    
    d={'sitenumber':new_sitenos,'Particle ID':sites,'numlocs':number_of_locs,'LENGTHtot':pathlengths}
    well_summ_parts=pd.DataFrame(data=d)
    
    d={'sitenumber':new_sitenos,'Particle ID':sites,'numlocs':number_of_locs,'var_res':VARs,'var_logres':logVARs,'var_res_sn':VARs_sn,'var_snlogres':VARs_snlogres,'RESeff':RESeffs,'snRESeff':snRESeffs,'logRESeff':logRESeffs,'snlogRESeff':snlogRESeffs}
    well_summ_parts=pd.DataFrame(data=d)
    well_summ_parts.loc[:,'logNormality']='xx'
    #after looping through all particles for a given well, test whether the effective resistivities are log normal.
    meanlog10reseff=np.nanmean(logRESeffs)
    well_summ_parts.loc[:,'meanlog10reseff']=meanlog10reseff

    # Goodness of fit test for effective resistivities of all the particles tracked from the well, to see if they have a lognormal distribution.
    loc,scale = meanlog10reseff, np.std(logRESeffs, ddof=1)
    cdf = stats.norm(loc, scale).cdf
    res=stats.ks_1samp(logRESeffs, cdf)
    ###print(res)
    if res.pvalue<.05:
        print('not log normal')
        well_summ_parts.loc[:,'logNormality']='not log normal'
    else:
        print('log normal')
        well_summ_parts.loc[:,'logNormality']='log normal'
    numparts=len(sites)
    print('number of particles:',numparts)
    numRESeffs=len(RESeffs)
    print('number of effective resistivities:',numRESeffs)
    print('\t')
    well_summ_partsshp=well_summ_parts.shape
    print('well_summ_parts:',well_summ_partsshp)

    # Calculating average variance of resistivity along a flowpath.
    smeanvar=np.nanmean(well_summ_parts['var_res'])
    ###########################++++++++++++++++++++++++++++++
    smeanlogvar=np.nanmean(well_summ_parts['var_logres'])
    smeanvar_sn=np.nanmean(well_summ_parts['var_res_sn'])
    smeanvar_logsn=np.nanmean(well_summ_parts['var_snlogres'])

    ameanreseff=np.nanmean(well_summ_parts['RESeff'])
    well_summ_parts.loc[:,'avg_resEff']=ameanreseff
    ameansnreseff=np.nanmean(well_summ_parts['snRESeff'])
    well_summ_parts.loc[:,'avg_snresEff']=ameansnreseff
    ameanlogreseff=np.nanmean(well_summ_parts['logRESeff'])
    well_summ_parts.loc[:,'avg_logresEff']=ameanlogreseff
    ameansnlogreseff=np.nanmean(well_summ_parts['snlogRESeff'])
    well_summ_parts.loc[:,'avg_snlogresEff']=ameansnlogreseff

    # Calculating variance of effective resistivities 
    well_summ_parts['dfms_reseff']=(well_summ_parts['RESeff']-ameanreseff)**2
    #VAR=(math.fsum(well_summ_parts['dfms_reseff']))/(numRESeffs-1)
    VAR=(np.nansum(well_summ_parts['dfms_reseff']))/(numRESeffs-1)
    well_summ_parts.loc[:,'VAR_resEff']=VAR#variance of effective resistivities of flowpaths to the well.

    # Calculating variance of sn effective res
    well_summ_parts['dfms_reseff_sn']=(well_summ_parts['snRESeff']-ameansnreseff)**2
    #VARsn=(math.fsum(well_summ_parts['dfms_reseff_sn']))/(numRESeffs-1)
    VARsn=(np.nansum(well_summ_parts['dfms_reseff_sn']))/(numRESeffs-1)
    well_summ_parts.loc[:,'VAR_snresEff']=VARsn#variance of sn effective resistivities of flowpaths to the well.

    #Calculating variance of effective log res
    well_summ_parts['dfms_logreseff']=(well_summ_parts['logRESeff']-ameanlogreseff)**2
    #VARlog10res=(math.fsum(well_summ_parts['dfms_logreseff']))/(numRESeffs-1)
    VARlog10res=(np.nansum(well_summ_parts['dfms_logreseff']))/(numRESeffs-1)
    well_summ_parts.loc[:,'VAR_logresEff']=VARlog10res#variance of effective log resistivities of flowpaths to the well.

    # Calculating variance of effective sn log res
    well_summ_parts['dfms_snlogreseff']=(well_summ_parts['snlogRESeff']-ameansnlogreseff)**2
    #VARsnlog10res=(math.fsum(well_summ_parts['dfms_snlogreseff']))/(numRESeffs-1)
    VARsnlog10res=(np.nansum(well_summ_parts['dfms_snlogreseff']))/(numRESeffs-1)
    well_summ_parts.loc[:,'VAR_snlogresEff']=VARsnlog10res#variance of sn effective log resistivities of flowpaths to the well.

    well_summ_parts.loc[:,'avgVar_res']=smeanvar#average of variances of resistivities along flowpaths to the well
    well_summ_parts.loc[:,'avgVar_logres']=smeanlogvar
    well_summ_parts.loc[:,'avgVar_snres']=smeanvar_sn
    well_summ_parts.loc[:,'avgVar_snlogres']=smeanvar_logsn
    well_summ_parts.loc[:,'avgRES_well']=meanres
    well_summ_parts.loc[:,'numparts_well']=numparts
    well_summ_parts.loc[:,'avglogRES_well']=meanlog10res
    #well_summ_parts.to_csv('well_summ_parts_{}_8284.csv'.format(new_sitenos[0]))
    #well_summ_parts.to_csv(os.path.join('welSumms','well_summ_parts_{}_954compare.csv'.format(siteno)))
    print('# file finished: ', idx + 1)
    print(f"File: {filename} has finished!")
    print('Grazie!')
    sys.exit()
    print("Thank you!")

print("All files have finished processing!")

filename: meras_2.2_volume_zones[350744090055601].mppth
siteno: 350744090055601
Processing file: meras_2.2_volume_zones[350744090055601].mppth
numparts: 200
Reprojection completed
max x_4326: -89.34260261502205
median x_4326: -90.1020383973075
mean x_4326: -90.05971870162313
min x_4326: -90.11795282539039
max y_4326: 35.200835123258294
median y_4326: 35.13119020330788
mean y_4326: 35.13271313697116
min y_4326: 35.034722133806646
plus1: nan
Start extraction.
botelv1int: -30
botelv1round: -30.0
botelv1roundn1: -30.0
botelv1round0: -30.0
botelv1round1: -30.0
botelv1str: -30.0
botelv1strrl2: -30
Extraction complete for file:  meras_2.2_volume_zones[350744090055601].mppth
Thank You!
Name: 1
nm2: 554
len(gx): 556
numlocs: 556
numlocs: 556
len(rom1): 556
length: 0
numlocs: 556
VR: -9999
cvr: -9999
RESeff: inf
Name: 2
nm2: 554
len(gx): 556
numlocs: 556
numlocs: 556
len(rom1): 556
length: 0
numlocs: 556
VR: -9999
cvr: -9999
RESeff: inf
Name: 3
nm2: 563
len(gx): 565
numlocs: 565
numlocs: 565
len

C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered

numlocs: 556
len(rom1): 556
length: 0
numlocs: 556
VR: -9999
cvr: -9999
RESeff: inf
Name: 16
nm2: 556
len(gx): 558
numlocs: 558
numlocs: 558
len(rom1): 558
length: 0
numlocs: 558
VR: -9999
cvr: -9999
RESeff: inf
Name: 17
nm2: 552
len(gx): 554
numlocs: 554
numlocs: 554
len(rom1): 554
length: 0
numlocs: 554
VR: -9999
cvr: -9999
RESeff: inf
Name: 18
nm2: 562
len(gx): 564
numlocs: 564
numlocs: 564
len(rom1): 564
length: 0
numlocs: 564
VR: -9999
cvr: -9999
RESeff: inf
Name: 19
nm2: 554
len(gx): 556
numlocs: 556
numlocs: 556
len(rom1): 556
length: 0
numlocs: 556
VR: -9999
cvr: -9999
RESeff: inf
Name: 20
nm2: 553
len(gx): 555
numlocs: 555
numlocs: 555
len(rom1): 555
length: 0
numlocs: 555
VR: -9999
cvr: -9999
RESeff: inf
Name: 21
nm2: 553
len(gx): 555
numlocs: 555
numlocs: 555
len(rom1): 555
length: 0
numlocs: 555
VR: -9999
cvr: -9999
RESeff: inf
Name: 22
nm2: 554
len(gx): 556
numlocs: 556
numlocs: 556
len(rom1): 556
length: 0
numlocs: 556
VR: -9999
cvr: -9999
RESeff: inf
Name: 23
nm2: 551
le

C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered

numlocs: 554
len(rom1): 554
length: 0
numlocs: 554
VR: -9999
cvr: -9999
RESeff: inf
Name: 45
nm2: 554
len(gx): 556
numlocs: 556
numlocs: 556
len(rom1): 556
length: 0
numlocs: 556
VR: -9999
cvr: -9999
RESeff: inf
Name: 46
nm2: 551
len(gx): 553
numlocs: 553
numlocs: 553
len(rom1): 553
length: 0
numlocs: 553
VR: -9999
cvr: -9999
RESeff: inf
Name: 47
nm2: 553
len(gx): 555
numlocs: 555
numlocs: 555
len(rom1): 555
length: 0
numlocs: 555
VR: -9999
cvr: -9999
RESeff: inf
Name: 48
nm2: 551
len(gx): 553
numlocs: 553
numlocs: 553
len(rom1): 553
length: 0
numlocs: 553
VR: -9999
cvr: -9999
RESeff: inf
Name: 49
nm2: 551
len(gx): 553
numlocs: 553
numlocs: 553
len(rom1): 553
length: 0
numlocs: 553
VR: -9999
cvr: -9999
RESeff: inf
Name: 50
nm2: 563
len(gx): 565
numlocs: 565
numlocs: 565
len(rom1): 565
length: 1
numlocs: 565
VR: -9999
cvr: -9999
RESeff: 1658.0947425432753
Name: 51
nm2: 554
len(gx): 556
numlocs: 556
numlocs: 556
len(rom1): 556
length: 0
numlocs: 556
VR: -9999
cvr: -9999
RESeff: inf
Name:

...

C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered

nm2: 555
len(gx): 557
numlocs: 557
numlocs: 557
len(rom1): 557
length: 0
numlocs: 557
VR: -9999
cvr: -9999
RESeff: inf
Name: 166
nm2: 555
len(gx): 557
numlocs: 557
numlocs: 557
len(rom1): 557
length: 0
numlocs: 557
VR: -9999
cvr: -9999
RESeff: inf
Name: 167
nm2: 562
len(gx): 564
numlocs: 564
numlocs: 564
len(rom1): 564
length: 1
numlocs: 564
VR: -9999
cvr: -9999
RESeff: 8130.31337032381
Name: 168
nm2: 562
len(gx): 564
numlocs: 564
numlocs: 564
len(rom1): 564
length: 0
numlocs: 564
VR: -9999
cvr: -9999
RESeff: inf
Name: 169
nm2: 558
len(gx): 560
numlocs: 560
numlocs: 560
len(rom1): 560
length: 0
numlocs: 560
VR: -9999
cvr: -9999
RESeff: inf
Name: 170
nm2: 562
len(gx): 564
numlocs: 564
numlocs: 564
len(rom1): 564
length: 1
numlocs: 564
VR: -9999
cvr: -9999
RESeff: 8720.386825361482
Name: 171
nm2: 563
len(gx): 565
numlocs: 565
numlocs: 565
len(rom1): 565
length: 4
numlocs: 565
VR: 14433042.668235475
cvr: 1.12297161802011
RESeff: 3383.063641461112
Name: 172
nm2: 555
len(gx): 557
numlocs: 5

C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered in double_scalars
  RESeff=RESeffnumerator/RESeffdenominator
C:\Users\mgratzer\AppData\Local\Temp\2\ipykernel_1068\1168408083.py:601: RuntimeWarning: divide by zero encountered

SystemExit: 

C:\Anaconda3\envs\gis-2\lib\site-packages\IPython\core\interactiveshell.py:3259: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


INDX2=range(504)
INDX2

test = pd.read_csv('particle_1_v2.csv')
grpdf_test = pd.DataFrame(data=test, index=INDX2)
grpdf_test

In [ ]:
#grpdf_test.to_csv('grpdf_1_test.csv', index=False)

test = pd.read_csv('grpdf_v2_1.csv')
VAR=(math.fsum(test['dfms']))/(496-1)  # Variance of raw resistivities along one particle's flowpath.
print("sum: ", math.fsum(test['dfms']))
print("Numlocs: ", 496)
print('VAR: ', VAR)

test

test2 = test.head(100)
test2

math.fsum(test2['dfms'])

VAR = (np.nansum(test['dfms'])/495)
VAR

type(VAR)

type(math.fsum(test2['dfms']))